# 01: Production Hybrid RAG: BM25 Sparse + Dense Vector Search + Reranking

**Track 13: Generative AI, LLMs, RAG & Multi-Agent Swarms** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Implement enterprise Hybrid Retrieval-Augmented Generation: BM25 lexical token search, dense vector embeddings, Reciprocal Rank Fusion (RRF), and Cross-Encoder reranking.


## 1. Load Knowledge Base & Document Chunking
Ingest enterprise documentation corpus and segment into semantic chunks.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = [
    "Tensorbox provides zero-setup Docker environments for AI and Machine Learning.",
    "PyTorch is an open source deep learning framework developed by Meta AI.",
    "FastAPI enables high-performance REST APIs with automatic OpenAPI Swagger documentation.",
    "BM25 is a ranking function used by search engines to estimate relevance of documents.",
    "Hybrid RAG combines dense vector retrieval with keyword BM25 indexing for superior accuracy.",
    "LoRA (Low-Rank Adaptation) reduces trainable parameters by decomposing weight update matrices."
]

print(f"Knowledge Corpus Chunks: {len(docs)}")

## 2. Reciprocal Rank Fusion (RRF) Hybrid Merger
Combine dense semantic ranks with sparse BM25 ranks:
$$\text{RRF}(d) = \sum_{m \in \mathcal{M}} \frac{1}{k + r_m(d)}$$

In [ ]:
def reciprocal_rank_fusion(sparse_ranks, dense_ranks, k=60):
    all_docs = set(sparse_ranks.keys()).union(set(dense_ranks.keys()))
    rrf_scores = {}
    for doc_id in all_docs:
        score = 0.0
        if doc_id in sparse_ranks:
            score += 1.0 / (k + sparse_ranks[doc_id])
        if doc_id in dense_ranks:
            score += 1.0 / (k + dense_ranks[doc_id])
        rrf_scores[doc_id] = score
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

query = "How does hybrid RAG work with BM25?"
sparse_ranking = {4: 1, 3: 2, 0: 3}
dense_ranking = {4: 1, 0: 2, 5: 3}

fusion_results = reciprocal_rank_fusion(sparse_ranking, dense_ranking)
print("=== Hybrid RRF Ranked Documents ===")
for doc_id, rrf_score in fusion_results:
    print(f"Doc #{doc_id} (RRF Score: {rrf_score:.5f}): {docs[doc_id]}")